# Statistical Investigation of Chronic Kidney Disease Risk Factors and Prediction
## Complex Engineering Project (CEP) — STA281 Spring 2026
### Team: Tehreek-e-Muhandis
---
**Dataset:** [Chronic Kidney Disease Dataset — Kaggle](https://www.kaggle.com/datasets/mansoordaku/ckdisease)

This notebook covers all four sections:
- **Section 1** — Dataset Understanding, Descriptive Statistics & Visualization (Q1–Q5)
- **Section 2** — Probability, Probability Distribution & CLT (Q6–Q12)
- **Section 3** — Confidence Interval & Hypothesis Testing (Q13–Q15)
- **Section 4** — Correlation, Regression & Healthcare Investigation (Q16–Q20)

# Library Import

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import binom, norm, uniform, expon
from scipy.stats import ttest_1samp
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.3
})
sns.set_theme(style='whitegrid', palette='muted')
print('All libraries imported successfully.')

All libraries imported successfully.


# Dataset Load

In [14]:

import pandas as pd
import os

dataset_path = "../dataset/kidney_disease.csv"

if os.path.exists(dataset_path):
    df_raw = pd.read_csv(dataset_path)
    print("Loaded dataset from local project directory successfully.")
else:
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}. "
        "Make sure kidney_disease.csv is inside the dataset folder."
    )
rename_map = {
    'id': 'ID',
    'age': 'Age',
    'bp': 'BloodPressure',
    'sg': 'SpecificGravity',
    'al': 'Albumin',
    'su': 'Sugar',
    'bgr': 'BloodGlucoseRandom',
    'bu': 'BloodUrea',
    'sc': 'SerumCreatinine',
    'sod': 'Sodium',
    'pot': 'Potassium',
    'hemo': 'Hemoglobin',
    'pcv': 'PackedCellVolume',
    'wc': 'WhiteBloodCellCount',
    'rc': 'RedBloodCellCount',
    'htn': 'Hypertension',
    'dm': 'DiabetesMellitus',
    'cad': 'CoronaryArteryDisease',
    'appet': 'Appetite',
    'pe': 'PedalEdema',
    'ane': 'Anemia',
    'classification': 'CKD'
}

df_raw.rename(
    columns={k: v for k, v in rename_map.items() if k in df_raw.columns},
    inplace=True
)

# Clean CKD target column
if 'CKD' in df_raw.columns and df_raw['CKD'].dtype == object:
    df_raw['CKD'] = (
        df_raw['CKD']
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            'ckd': 1,
            'notckd': 0
        })
    )

num_cols = [
    'Age',
    'BloodPressure',
    'SpecificGravity',
    'Albumin',
    'Sugar',
    'BloodGlucoseRandom',
    'BloodUrea',
    'SerumCreatinine',
    'Sodium',
    'Potassium',
    'Hemoglobin',
    'PackedCellVolume',
    'WhiteBloodCellCount',
    'RedBloodCellCount'
]

for col in num_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

Loaded dataset from local project directory successfully.
Dataset shape: (400, 26)


,ID,Age,BloodPressure,SpecificGravity,Albumin,Sugar,rbc,pc,pcc,ba,...,PackedCellVolume,WhiteBloodCellCount,RedBloodCellCount,Hypertension,DiabetesMellitus,CoronaryArteryDisease,Appetite,PedalEdema,Anemia,CKD
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


# SECTION 1 — Dataset Understanding, Descriptive Statistics & Visualization
## Question 1 — Dataset Structure & Variable Identification

In [15]:
print('=' * 65)
print('QUESTION 1 — Dataset Structure')
print('=' * 65)

print(f'\n1. Number of observations (rows) : {df_raw.shape[0]}')
print(f'2. Number of variables (columns) : {df_raw.shape[1]}')

print('\n3. Variable Names & Data Types:')
dtype_df = pd.DataFrame({
    'Variable': df_raw.columns,
    'Data Type': df_raw.dtypes.values,
    'Non-Null Count': df_raw.notnull().sum().values,
    'Null Count': df_raw.isnull().sum().values
})
print(dtype_df.to_string(index=False))

print('\n--- Healthcare Interpretation ---')
print('The dataset contains 400 patient records with 25 variables.')
print('Key continuous variables include SerumCreatinine, BloodUrea, Hemoglobin,')
print('and BloodPressure — each critical for assessing kidney function.')
print('The CKD target variable is binary (1=CKD, 0=No CKD).')
print('Missing values are present and must be handled before analysis.')

QUESTION 1 — Dataset Structure

1. Number of observations (rows) : 400
2. Number of variables (columns) : 26

3. Variable Names & Data Types:
             Variable Data Type  Non-Null Count  Null Count
                   ID     int64             400           0
                  Age   float64             391           9
        BloodPressure   float64             388          12
      SpecificGravity   float64             353          47
              Albumin   float64             354          46
                Sugar   float64             351          49
                  rbc       str             248         152
                   pc       str             335          65
                  pcc       str             396           4
                   ba       str             396           4
   BloodGlucoseRandom   float64             356          44
            BloodUrea   float64             381          19
      SerumCreatinine   float64             383          17
               Sod